# QCB faculty map: dimensionality reduction and clustering

Each of the 48 core, joint and emeritus QCB faculty is described by three feature vectors built from their papers (see `scripts/build_features.py` and `data/features/features-report.md`):

| Representation | What it is | Size |
|---|---|---|
| `tfidf` | Words and two-word phrases from titles, abstracts and keywords, weighted by how few faculty use them | 48 × ~26,000 |
| `topics` | Share of their work in each OpenAlex topic (about 1,500 topics appear) | 48 × ~1,500 |
| `specter` | Mean SPECTER2 embedding of their papers (a language model trained on scientific citations) | 48 × 768 |

The goal is a 2-D map where nearby faculty do similar research, plus clusters to colour it by. This notebook gives each method a working baseline. Each one needs a judgement call that's yours to make, and the **Try:** prompts point at them.

One thing shapes every choice here: **48 points is a small dataset.** Methods built for thousands of points (t-SNE, UMAP) need their neighbourhood settings turned down, and there's enough room to look at every point by name.

At the end, `export_layout` writes the chosen coordinates and clusters to `data/layout.json` for the interactive map.

**Before running:** the features and edges are committed, so a fresh clone can run this as is. To rebuild them: `python3 scripts/fetch_papers.py`, `python3 scripts/build_edges.py`, then `.venv/bin/python scripts/build_features.py`.

In [ ]:
import json
import pathlib
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from scipy.spatial import procrustes
from scipy.spatial.distance import squareform
from scipy.stats import spearmanr
from sklearn.decomposition import PCA
from sklearn.manifold import MDS, TSNE, trustworthiness
from sklearn.metrics import adjusted_rand_score, silhouette_score

ROOT = pathlib.Path.cwd()
if not (ROOT / "data" / "features").exists():  # opened from the repo root instead
    ROOT = ROOT / "faculty-map"
DATA = ROOT / "data"

F = np.load(DATA / "features" / "faculty-features.npz", allow_pickle=False)
roster = {p["id"]: p for p in json.loads((DATA / "faculty.json").read_text())["faculty"]}
edges = json.loads((DATA / "edges.json").read_text())

plt.rcParams.update({"figure.dpi": 110, "font.size": 9})
warnings.filterwarnings("ignore", module="umap")  # notes about precomputed distances and seeds
print("features:", ", ".join(k for k in F.files if F[k].ndim == 2))

## Who is in the data

`fac` has one row per person. `area` is a coarse grouping of each person's *home* department (core faculty are "QCB"), used to colour plots. It's a useful contrast: does research similarity follow departments, or cut across them?

Emeriti are included by default; set `INCLUDE_EMERITI = False` to leave them out of everything below.

In [ ]:
INCLUDE_EMERITI = True

AREAS = {
    "Quantitative and Computational Biology": "QCB",
    "Population and Public Health Sciences": "Public health genetics",
    "Physics and Astronomy": "Physics & mechanics",
    "Aerospace and Mechanical Engineering": "Physics & mechanics",
    "Computer Science": "Computer science",
    "Industrial and Systems Engineering": "Computer science",
    "Biological Sciences": "Biological sciences",
    "Biomedical Engineering": "Biomedical engineering",
    "Cancer Biology": "Medicine & cancer",
    "Medicine": "Medicine & cancer",
    "Neurology": "Neuroscience & aging",
    "Ophthalmology": "Neuroscience & aging",
    "Gerontology": "Neuroscience & aging",
    "Chemistry": "Chemistry",
}

ids = list(F["faculty_ids"])
fac = pd.DataFrame([roster[i] for i in ids])[["id", "name", "first", "last", "group", "home_department"]]
fac["area"] = fac["home_department"].map(AREAS).fillna("Other")
fac["n_papers"] = F["n_papers"]
# Short labels: last name, with an initial where two people share one (the two Lius).
dupes = fac["last"].duplicated(keep=False)
fac["label"] = np.where(dupes, fac["first"].str[0] + ". " + fac["last"], fac["last"])

keep = (fac["group"] != "emeritus").to_numpy() | INCLUDE_EMERITI
fac = fac[keep].reset_index(drop=True)
print(len(fac), "faculty;", fac["group"].value_counts().to_dict())
fac.head()

## The three representations

Each needs a little preprocessing before distances mean anything:

- **TF-IDF** rows are already unit length, so cosine similarity is a dot product.
- **Topics** are probability distributions. The square root (the *Hellinger* transform) turns them into unit vectors whose dot product measures how much two distributions overlap, and keeps a few huge topics from dominating.
- **SPECTER2** vectors all point roughly the same way: raw cosine similarities between unrelated papers are around 0.8. Subtracting the mean vector (*centring*) removes that shared direction so the differences stand out.

`REP` picks the representation used in the rest of the notebook.

In [ ]:
def unit_rows(X):
    X = np.asarray(X, dtype=float)
    n = np.linalg.norm(X, axis=1, keepdims=True)
    return X / np.where(n == 0, 1, n)


X_all = {
    "tfidf": unit_rows(F["tfidf"][keep]),
    "topics": unit_rows(np.sqrt(F["topics"][keep])),
    "specter": unit_rows(F["specter"][keep] - F["specter"][keep].mean(axis=0)),
}

# Change the representation here only: X, S and D below are what everything else uses.
REP = "specter"   # Try: "tfidf" or "topics"
X = X_all[REP]
S = X @ X.T                      # cosine similarity
D = np.clip(1 - S, 0, None)      # cosine distance
np.fill_diagonal(D, 0)
print(REP, X.shape)

### Helpers

`scatter` draws a layout with names. `align` rotates/reflects one layout onto another (Procrustes), since t-SNE, UMAP and MDS outputs have no fixed orientation and are hard to compare otherwise. `knn_kept` measures what a 2-D layout preserves: for each person, how many of their *k* nearest neighbours in the full feature space are still among their *k* nearest on the map.

In [ ]:
AREA_COLORS = dict(zip(sorted(fac["area"].unique()), sns.color_palette("tab10")))
MARKERS = {"core": "o", "joint": "s", "emeritus": "^"}


def scatter(xy, ax=None, color_by="area", title="", show_edges=False, labels=True, legend=True):
    if ax is None:
        _, ax = plt.subplots(figsize=(8, 7))
    if show_edges:
        draw_edges(xy, ax)
    by_area = isinstance(color_by, str)
    if by_area:
        colors = fac["area"].map(AREA_COLORS)
    else:  # an array of cluster numbers
        pal = sns.color_palette("tab10", int(np.max(color_by)) + 1)
        colors = [pal[c] for c in color_by]
    colors = np.asarray(list(colors), dtype=object)
    for g, m in MARKERS.items():
        idx = (fac["group"] == g).to_numpy()
        if idx.any():
            ax.scatter(xy[idx, 0], xy[idx, 1], c=list(colors[idx]), marker=m, s=46,
                       edgecolor="white", linewidth=0.6, zorder=3, label=g)
    if labels:
        for (x, y), lab in zip(xy, fac["label"]):
            ax.annotate(lab, (x, y), xytext=(3, 3), textcoords="offset points", fontsize=7, zorder=4)
    if legend and by_area:
        handles = [plt.Line2D([], [], marker="o", ls="", color=c, label=a) for a, c in AREA_COLORS.items()]
        handles += [plt.Line2D([], [], marker=m, ls="", color="black", markerfacecolor="none", label=g)
                    for g, m in MARKERS.items()]
        ax.legend(handles=handles, fontsize=7, loc="upper left", bbox_to_anchor=(1.0, 1.0), frameon=False)
    ax.set_title(title)
    ax.set_xticks([]); ax.set_yticks([])
    return ax


def draw_edges(xy, ax, min_strength=0.0):
    pos = {pid: k for k, pid in enumerate(fac["id"])}
    for e in edges["edges"]:
        a, b = pos.get(e["source"]), pos.get(e["target"])
        if a is None or b is None or e["strength"] < min_strength:
            continue
        w = 0.4 + 1.2 * np.log10(1 + 10 * e["strength"])   # log scale: strengths run 0.01–100
        ax.plot(*xy[[a, b]].T, color="gray", alpha=0.45, lw=w, zorder=1)


def align(xy, ref):
    # Procrustes standardizes both; rescale the aligned layout back to ref's spread.
    _, aligned, _ = procrustes(ref, xy)
    return aligned * ref.std() / aligned.std() + ref.mean(axis=0)


def knn_kept(D_high, xy, k=5):
    D_low = np.linalg.norm(xy[:, None] - xy[None], axis=-1)
    hits = []
    for i in range(len(xy)):
        hi = set(np.argsort(D_high[i])[1:k + 1])
        lo = set(np.argsort(D_low[i])[1:k + 1])
        hits.append(len(hi & lo) / k)
    return float(np.mean(hits))

## 1. Look at the similarities first

Before squashing anything into 2-D, look at the full similarity matrix. The clustermap reorders rows and columns so similar people sit together; blocks along the diagonal are research communities.

**Try:** switch `REP` above and rerun from here. Which communities show up in all three? Where do the representations disagree?

In [ ]:
lab = fac["label"].tolist()
g = sns.clustermap(pd.DataFrame(S, index=lab, columns=lab), method="average",
                   metric="cosine", cmap="mako", figsize=(10, 10), dendrogram_ratio=0.12,
                   xticklabels=True, yticklabels=True, cbar_pos=(0.02, 0.85, 0.02, 0.1))
g.ax_heatmap.tick_params(labelsize=6)
g.fig.suptitle(f"Cosine similarity ({REP})", y=1.01);

### Do the representations agree?

Spearman correlation between the similarity matrices (over all pairs): 1 means they rank who-is-similar-to-whom identically. The scatter shows where two representations part ways.

In [ ]:
iu = np.triu_indices(len(fac), 1)
sims = {k: (v @ v.T)[iu] for k, v in X_all.items()}
print(pd.DataFrame({a: {b: round(spearmanr(sims[a], sims[b])[0], 2) for b in sims} for a in sims}))

fig, ax = plt.subplots(figsize=(5, 4.5))
ax.scatter(sims["tfidf"], sims["specter"], s=6, alpha=0.5)
ax.set_xlabel("TF-IDF similarity"); ax.set_ylabel("SPECTER2 similarity (centred)")
# Label the pairs where the two disagree most (in rank).
gap = np.argsort(sims["tfidf"]).argsort() - np.argsort(sims["specter"]).argsort()
for k in np.r_[np.argsort(gap)[:4], np.argsort(gap)[-4:]]:
    i, j = iu[0][k], iu[1][k]
    ax.annotate(f"{lab[i]}–{lab[j]}", (sims["tfidf"][k], sims["specter"][k]), fontsize=6)

## 2. PCA

PCA finds the directions of greatest variance: a linear, global method, and the natural baseline. With 48 points there can be at most 47 components.

For TF-IDF the components are interpretable: each one is a weighted list of terms, so you can read off what PC1 and PC2 *mean*.

**Try:** How much variance do two components capture? Read the top terms of PC1 and PC2 (run with `REP = "tfidf"`). Is PC1 something like molecular ↔ organismal scale, or methods ↔ biology?

In [ ]:
pca = PCA(n_components=min(20, len(fac) - 1), random_state=0)
Z_pca = pca.fit_transform(X)


fig, axes = plt.subplots(1, 2, figsize=(14, 6), gridspec_kw={"width_ratios": [1, 2.2]})
axes[0].bar(range(1, len(pca.explained_variance_ratio_) + 1), pca.explained_variance_ratio_)
axes[0].plot(range(1, len(pca.explained_variance_ratio_) + 1), np.cumsum(pca.explained_variance_ratio_), "k.-")
axes[0].set_xlabel("component"); axes[0].set_ylabel("variance explained (bars) / cumulative")
scatter(Z_pca[:, :2], ax=axes[1], title=f"PCA ({REP}): PC1 vs PC2")

if REP == "tfidf":
    vocab = F["tfidf_vocab"]
    for c in range(2):
        order = np.argsort(pca.components_[c])
        print(f"PC{c + 1} +:", ", ".join(vocab[order[-10:]][::-1]))
        print(f"PC{c + 1} −:", ", ".join(vocab[order[:10]]))

## 3. MDS

Metric MDS places points so their 2-D distances match the cosine distances as closely as possible (it minimizes *stress*). Unlike PCA it works from distances alone, and unlike t-SNE/UMAP it tries to keep **global** distances, so it's honest about how far apart the clusters are.

**Try:** Rerun with a different `random_state`: MDS has local optima. How does the stress compare across representations?

In [ ]:
mds = MDS(n_components=2, dissimilarity="precomputed", metric=True, n_init=8,
          random_state=0, normalized_stress="auto")
Z_mds = mds.fit_transform(D)
# Kruskal's stress-1: 0 is perfect; under ~0.1 is good, over ~0.2 means a lot is distorted.
iu = np.triu_indices(len(fac), 1)
d_map = np.linalg.norm(Z_mds[:, None] - Z_mds[None], axis=-1)[iu]
print("stress-1:", round(float(np.sqrt(((D[iu] - d_map) ** 2).sum() / (D[iu] ** 2).sum())), 3))
scatter(Z_mds, title=f"MDS on cosine distance ({REP})");

## 4. t-SNE

t-SNE keeps **local** neighbourhoods: who is next to whom. The distances *between* clusters and the cluster sizes don't mean much. *Perplexity* is roughly the number of neighbours each point pays attention to. The default (30) is far too big for 48 points (sklearn requires it to be below the number of samples), so here it's swept from 4 to 15.

**Try:** Which perplexity gives stable, sensible neighbourhoods? Change `random_state`: does the picture survive?

In [ ]:
perplexities = [4, 8, 15]
fig, axes = plt.subplots(1, len(perplexities), figsize=(18, 6))
Z_tsne = {}
for ax, p in zip(axes, perplexities):
    Z_tsne[p] = TSNE(n_components=2, perplexity=p, metric="precomputed", init="random",
                     random_state=0).fit_transform(D)
    scatter(align(Z_tsne[p], Z_mds), ax=ax, title=f"t-SNE perplexity {p}", legend=(p == perplexities[0]))

## 5. UMAP

UMAP builds a nearest-neighbour graph and lays it out. Like t-SNE it emphasizes local structure, though it tends to keep a bit more of the global arrangement. `n_neighbors` plays the role of perplexity; `min_dist` controls how tightly points pack.

**Try:** Sweep `n_neighbors` and `min_dist`. When does the map fragment into islands, and when does it blur into one blob?

In [ ]:
import umap  # slow-ish first import (numba compiles)

neighbors = [4, 6, 8, 15]
fig, axes = plt.subplots(1, len(neighbors), figsize=(6 * len(neighbors), 6))
Z_umap = {}
for ax, k in zip(axes, neighbors):
    Z_umap[k] = umap.UMAP(n_neighbors=k, min_dist=0.3, metric="precomputed",
                          random_state=0).fit_transform(D)
    scatter(align(Z_umap[k], Z_mds), ax=ax, title=f"UMAP n_neighbors={k}", legend=(k == neighbors[0]))

## 6. Compare the layouts

Two scores of how faithful each 2-D layout is to the full feature space:

- **trustworthiness** (sklearn): are a point's neighbours on the map really its neighbours? Penalizes false neighbours.
- **kNN kept**: the share of each person's true 5 nearest neighbours that are still among their 5 nearest on the map.

**Try:** Is there a layout that wins on both? Does the winner change with `REP`? Faithful neighbours and readable global structure pull against each other. Which matters more for a map students will browse?

In [ ]:
layouts = {"PCA": Z_pca[:, :2], "MDS": Z_mds}
layouts.update({f"t-SNE p={p}": z for p, z in Z_tsne.items()})
layouts.update({f"UMAP k={k}": z for k, z in Z_umap.items()})

scores = pd.DataFrame({
    name: {"trustworthiness (k=5)": trustworthiness(D, z, n_neighbors=5, metric="precomputed"),
           "kNN kept (k=5)": knn_kept(D, z, k=5)}
    for name, z in layouts.items()
}).T.round(3)
scores

## 7. Clustering

Agglomerative (hierarchical) clustering on the cosine distances, with average linkage. The dendrogram shows the whole hierarchy; cutting it at *k* clusters gives labels. The silhouette score (−1 to 1) measures how much closer each person is to their own cluster than to the next one. It's a guide, not an oracle.

**Try:** Compare `method="average"`, `"complete"` and `"ward"` (ward needs Euclidean distances: use `linkage(X, "ward")`). Pick `K` by eye *and* by silhouette. Then read the cluster descriptions below: would you give each cluster a name?

In [ ]:
Lk = linkage(squareform(D, checks=False), method="average")

fig, axes = plt.subplots(1, 2, figsize=(16, 5), gridspec_kw={"width_ratios": [2.4, 1]})
dendrogram(Lk, labels=fac["label"].tolist(), leaf_font_size=7, ax=axes[0], color_threshold=None)
axes[0].set_title(f"Average-linkage dendrogram ({REP})")
ks = range(2, 13)
sil = [silhouette_score(D, fcluster(Lk, k, "maxclust"), metric="precomputed") for k in ks]
axes[1].plot(list(ks), sil, "o-"); axes[1].set_xlabel("k"); axes[1].set_ylabel("silhouette")

In [ ]:
K = 9   # Try: choose from the dendrogram and the silhouette curve
clusters = fcluster(Lk, K, "maxclust") - 1

print("agreement with home-department areas (adjusted Rand index):",
      round(adjusted_rand_score(fac["area"], clusters), 2))
scatter(Z_mds, color_by=clusters, title=f"MDS coloured by {K} clusters")

# What each cluster is about: terms and topics most over-represented relative to everyone.
T = F["tfidf"][keep]; P = F["topics"][keep]
for c in range(K):
    members = clusters == c
    terms = F["tfidf_vocab"][np.argsort(-(T[members].mean(0) - T.mean(0)))[:8]]
    topics = F["topic_names"][np.argsort(-(P[members].mean(0) - P.mean(0)))[:2]]
    print(f"\n[{c}] {', '.join(fac.loc[members, 'label'])}")
    print("    terms: ", ", ".join(terms))
    print("    topics:", "; ".join(topics))

## 8. Collaborations on the map

The map's positions come from research similarity; the lines come from co-authored papers (`data/edges.json`, width on a log scale). The two are built from different evidence, so laying one over the other asks a real question: **do people who work on similar things actually work together?**

The test below compares the average feature distance of collaborating pairs with randomly chosen pairs. The p-value is the share of 10,000 random sets of pairs that are at least as close.

**Try:** Which collaborations *bridge* clusters? Who is similar to many colleagues but collaborates with none of them (a potential introduction)?

In [ ]:
scatter(Z_mds, show_edges=True, title=f"MDS ({REP}) with collaboration edges")

pos = {pid: k for k, pid in enumerate(fac["id"])}
pairs = [(pos[e["source"]], pos[e["target"]]) for e in edges["edges"]
         if e["source"] in pos and e["target"] in pos]
observed = np.mean([D[a, b] for a, b in pairs])
rng = np.random.default_rng(0)
null = []
for _ in range(10_000):
    a = rng.integers(0, len(fac), len(pairs)); b = rng.integers(0, len(fac), len(pairs))
    ok = a != b
    null.append(D[a[ok], b[ok]].mean())
null = np.array(null)
p = np.mean(null <= observed)
print(f"{len(pairs)} collaborating pairs: mean distance {observed:.3f} vs random {null.mean():.3f}; "
      + (f"p = {p:.4f}" if p > 0 else f"p < {1 / len(null):.4f} (no random set was as close)"))

## 9. Export the map data

`export_layout` writes `data/layout.json`, which the interactive map will read:

- **Positions:** one set of coordinates per entry in `LAYOUTS`, all from the representation chosen by `REP`. Each is rotated to line up with the MDS layout, as in the plots above, so the map can move smoothly between views. Coordinates run 0–1 with the same scale on both axes, so shapes aren't stretched.
- **Tree and clusters:** the average-linkage tree on `TREE_REP`, cut into `TREE_K` clusters. The whole tree is saved (`merges`, with `leaf_order` for drawing), so the map can redraw the dendrogram or let viewers pick the number of clusters.

`TREE_REP` is separate from `REP` so positions and colours can come from different representations.

In [ ]:
EXPORT = True

LAYOUTS = {
    "tsne": {"name": "t-SNE", "settings": {"perplexity": 4}, "xy": Z_tsne[4]},
    "umap": {"name": "UMAP", "settings": {"n_neighbors": 6, "min_dist": 0.3}, "xy": Z_umap[6]},
}
TREE_REP = REP   # or name a different representation, e.g. "tfidf"
TREE_K = K


def unit_square(xy):
    """Fit into 0-1 with one scale for both axes, centred."""
    xy = np.asarray(xy, dtype=float)
    lo, hi = xy.min(0), xy.max(0)
    return (xy - (lo + hi) / 2) / (hi - lo).max() + 0.5


def export_layout(path=DATA / "layout.json"):
    assert X is X_all[REP], "REP doesn't match X: set REP in the cell that builds X, then rerun from there"
    Xt = X_all[TREE_REP]
    Dt = np.clip(1 - Xt @ Xt.T, 0, None)
    np.fill_diagonal(Dt, 0)
    tree = linkage(squareform(Dt, checks=False), method="average")
    cut = fcluster(tree, TREE_K, "maxclust") - 1
    coords = {key: unit_square(align(spec["xy"], Z_mds)) for key, spec in LAYOUTS.items()}

    people = []
    for i, row in fac.iterrows():
        person = {"id": row["id"], "label": row["label"], "cluster": int(cut[i])}
        person.update({key: [round(float(v), 4) for v in xy[i]] for key, xy in coords.items()})
        people.append(person)
    out = {
        "representation": REP,
        "include_emeriti": INCLUDE_EMERITI,
        "layouts": {key: {"name": s["name"], "settings": s["settings"]} for key, s in LAYOUTS.items()},
        "tree": {
            "representation": TREE_REP,
            "linkage": "average",
            "distance": "cosine",
            "k": TREE_K,
            "silhouette": round(float(silhouette_score(Dt, cut, metric="precomputed")), 3),
            # Node numbers: 0..n-1 are faculty in the order below, n+i is merges[i].
            "merges": [{"left": int(a), "right": int(b), "height": round(float(h), 4), "size": int(n)}
                       for a, b, h, n in tree],
            "leaf_order": [fac["id"][i] for i in dendrogram(tree, no_plot=True)["leaves"]],
        },
        "faculty": people,
    }
    path.write_text(json.dumps(out, indent=1, ensure_ascii=False) + "\n", encoding="utf-8")
    print(f"wrote {path}: {len(people)} faculty, {len(coords)} layouts on {REP}, "
          f"{TREE_K} clusters on {TREE_REP} (silhouette {out['tree']['silhouette']})")
    for c in range(TREE_K):
        print(f"  [{c}]", ", ".join(fac.loc[cut == c, "label"]))


if EXPORT:
    export_layout()